In [1]:
// Parameters
var BATCH_MODE = "true";

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Aspire AppHost : orchestrer un frontend Vite/pnpm

Ce notebook est le **dixième volet** de la série Aspire (volets [01-09](README.md) livrés). Il traite l'axe différé par le volet CSharpRepl ([#10897](https://github.com/jsboige/CoursIA/issues/10897)) : **l'orchestration d'une ressource frontend dans l'AppHost**, l'endroit où une stack .NET rencontre habituellement le monde JavaScript — et où elle est le plus souvent renvoyée à un terminal séparé (`pnpm dev` à la main, hors modèle de ressources).

**Question du volet** : que reste-t-il à faire à la main quand le frontend devient une *ressource déclarée* comme les autres — même cycle de vie, même dashboard, même arrêt propre — et que coûte ce statut par rapport au `docker-compose` qui orchestre les services de la pile GenAI ?

Le lab réel vit dans [`FrontendLab/`](FrontendLab/) : un AppHost file-based (`apphost.cs` + `aspire.config.json`, même modèle que [`GenAiStack.AppHost/`](GenAiStack.AppHost/)) qui orchestre le **vrai** dev-server Vite lancé par le **vrai** gestionnaire pnpm — aucune réimplémentation, aucun serveur de substitution.

**Durée** : ~15-20 min. **Prérequis** : .NET SDK 10 (user-local), CLI Aspire (`dotnet tool install -g aspire`), pnpm (`npm install -g pnpm`).

In [2]:
// Helper partagé entre les cellules (.NET Interactive : un static class persiste).
// Cellule = DECLARATION SEULE : en C# top-level, aucune instruction ne peut
// suivre une declaration de type dans la meme unite de compilation.
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Net.Http;
using System.Net.Sockets;
using System.Text.RegularExpressions;
using System.Threading;

public static class LabShell
{
    public static readonly string AspireCmd = Path.Combine(
        Environment.GetFolderPath(Environment.SpecialFolder.UserProfile),
        ".dotnet", "tools", "aspire.cmd");              // CLI Aspire (dotnet tool global)
    public static readonly string PnpmCmd = Path.Combine(
        Environment.GetFolderPath(Environment.SpecialFolder.ApplicationData),
        "npm", "pnpm.cmd");                              // pnpm (npm -g, bin user-scope)
    public static readonly string LabDir = FindLabDir();
    public static readonly string FrontendDir = Path.Combine(LabDir, "frontend");
    public static string LabLogPath = "";                // journal CLI du run courant (cellule demarrage)

    public static string FindLabDir()
    {
        var dir = new DirectoryInfo(Directory.GetCurrentDirectory());
        while (dir != null && !Directory.Exists(Path.Combine(dir.FullName, "MyIA.AI.Notebooks")))
            dir = dir.Parent;
        return Path.Combine(dir!.FullName, "MyIA.AI.Notebooks", "GenAI",
            "Integrations-DotNet", "Aspire", "FrontendLab");
    }

    // Sur ce depot les SDK .NET ne vivent que dans le dotnet user-local (mesure :
    // %ProgramFiles%/dotnet/sdk vide, %LOCALAPPDATA%/Microsoft/dotnet/sdk porte 8/9/10).
    // Les process lances ici heritent sinon d'un PATH sans SDK : on le pose explicitement.
    private static readonly string UserDotnet = Path.Combine(
        Environment.GetFolderPath(Environment.SpecialFolder.LocalApplicationData), "Microsoft", "dotnet");

    // Execute une commande, capture stdout+stderr et le code de sortie. Un .cmd EXIGE
    // cmd.exe (Windows) : CreateProcess ne resout pas les shims .cmd -- piege mesure au §1.
    public static (int rc, string text) RunEx(string fileName, string arguments, string workingDir, int timeoutSeconds = 180)
    {
        var useCmd = fileName.EndsWith(".cmd");
        var psi = new ProcessStartInfo(useCmd ? "cmd.exe" : fileName,
                                      useCmd ? $"/c {Path.GetFileName(fileName)} {arguments}" : arguments)
        {
            WorkingDirectory = workingDir,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            UseShellExecute = false,
        };
        psi.EnvironmentVariables["PATH"] =
            UserDotnet + ";" + Path.GetDirectoryName(PnpmCmd) + ";" + psi.EnvironmentVariables["PATH"];
        psi.EnvironmentVariables["DOTNET_ROOT"] = UserDotnet;
        using var proc = Process.Start(psi)!;
        string output = proc.StandardOutput.ReadToEnd();
        string err = proc.StandardError.ReadToEnd();
        if (!proc.WaitForExit(timeoutSeconds * 1000)) { proc.Kill(); throw new TimeoutException($"{fileName} {arguments}"); }
        var text = output + (err.Trim().Length > 0 ? "\n[stderr] " + err.Trim() : "");
        return (proc.ExitCode, Regex.Replace(text, @"\x1b\[[0-9;]*m|\x1b\][^\x1b]*\x1b\\", ""));   // ANSI couleurs + hyperliens
    }

    public static string Run(string fileName, string arguments, string workingDir, int timeoutSeconds = 180)
        => RunEx(fileName, arguments, workingDir, timeoutSeconds).text;

    // --nologo : sortie redirigee, la banniere de premier usage de la CLI (Aspire.Cli 13.4.6)
    // plante et sa sentinelle n'est jamais posee (lecon du volet Orleans/02).
    public static string RunAspire(string arguments, int timeoutSeconds = 180)
        => Run(AspireCmd, $"{arguments} --nologo", LabDir, timeoutSeconds);

    // Attendre qu'un port TCP local accepte une connexion (dev-server Vite : ~5 s).
    public static bool WaitForPort(int port, int timeoutSeconds = 60)
    {
        var deadline = DateTime.UtcNow.AddSeconds(timeoutSeconds);
        while (DateTime.UtcNow < deadline)
        {
            try
            {
                using var tcp = new TcpClient();
                var connect = tcp.BeginConnect("127.0.0.1", port, null, null);
                if (connect.AsyncWaitHandle.WaitOne(1000) && tcp.Connected) return true;
            }
            catch { /* pas encore pret */ }
            Thread.Sleep(800);
        }
        return false;
    }
}

In [3]:
// Versions reelles des outils du lab (garde SOTA : ce qui va s'executer est installe).
var vAspire = LabShell.Run(LabShell.AspireCmd, "--version", LabShell.LabDir, 60);
var vPnpm   = LabShell.Run(LabShell.PnpmCmd, "--version", LabShell.FrontendDir, 60);
var vNode   = LabShell.Run("node", "--version", LabShell.FrontendDir, 60);
var cheminVite = Path.Combine(LabShell.FrontendDir, "node_modules", "vite", "package.json");
var viteInstallee = File.Exists(cheminVite)
    ? "installee (" + Regex.Match(File.ReadAllText(cheminVite), "\"version\": \"([^\"]+)\"").Groups[1].Value + ")"
    : "absente (installee cellule §2)";
Console.WriteLine($"[aspire] {vAspire.Trim()}");
Console.WriteLine($"[pnpm]   {vPnpm.Trim()}");
Console.WriteLine($"[node]   {vNode.Trim()}");
Console.WriteLine($"[vite]   {viteInstallee}");
Console.WriteLine("[lab]    MyIA.AI.Notebooks/GenAI/Integrations-DotNet/Aspire/FrontendLab");

[aspire] 13.6.2+af07e33f8d1ef733645138b3f8e36067d073148d


[pnpm]   12.15.0


[node]   v24.20.0


[vite]   installee (7.3.7)


[lab]    MyIA.AI.Notebooks/GenAI/Integrations-DotNet/Aspire/FrontendLab


### Lecture de l'environnement

Trois outils réels hors du dépôt : la CLI Aspire (l'orchestrateur), pnpm (le gestionnaire de paquets du frontend), node (le runtime du dev-server). Vite est la dépendance du lab — sa version lue ci-dessus vient de `node_modules/vite/package.json`, pas d'une déclaration. Si l'une des lignes échoue : installer l'outil (règle F du dépôt — réparer, jamais contourner), pas de repli sur un serveur de substitution.

## 1. Le lab : quatre pièces, aucune réimplémentation

| Fichier | Rôle |
|---|---|
| [`FrontendLab/apphost.cs`](FrontendLab/apphost.cs) | L'AppHost file-based : **une** ressource `frontend` + **une** ligne d'endpoint |
| [`FrontendLab/aspire.config.json`](FrontendLab/aspire.config.json) | Pointe la CLI vers `apphost.cs` (modèle SDK file-based, comme `GenAiStack.AppHost/`) |
| [`FrontendLab/frontend/package.json`](FrontendLab/frontend/package.json) | Le vrai projet Vite : script `dev`, dépendance `vite` |
| [`FrontendLab/frontend/vite.config.js`](FrontendLab/frontend/vite.config.js) | `host 127.0.0.1`, `port 5173`, **`strictPort`** (le port ne dérive pas : la sonde du carnet reste déterministe) |

Le frontend lui-même est volontairement minimal (`index.html` + `src/main.js`) : le grain porte **l'orchestration**, pas la richesse du front.

In [4]:
// Le source complet de l'AppHost, et l'inventaire du frontend (chemins affiches relatifs au lab).
Console.WriteLine("=== FrontendLab/apphost.cs ===");
Console.WriteLine(File.ReadAllText(Path.Combine(LabShell.LabDir, "apphost.cs")).Trim());
Console.WriteLine("\n=== fichiers du frontend (hors node_modules) ===");
foreach (var f in Directory.GetFiles(LabShell.FrontendDir, "*", SearchOption.AllDirectories)
                           .Where(p => !p.Contains("node_modules")))
    Console.WriteLine("  " + Path.GetRelativePath(LabShell.LabDir, f));

=== FrontendLab/apphost.cs ===


#:sdk Aspire.AppHost.Sdk@13.4.6
using Aspire.Hosting;

// AppHost Aspire orchestrant une RESSOURCE FRONTEND Vite/pnpm -- grain #22363,
// axe Part 2 differe par #10897 (EPIC #10473 *The Unexpected AI Stack*).
//
// Choix de l'API, MESURE NuGet + runtime a la redaction (2026-10-24) :
//
//  1. Aspire.Hosting boxed 13.4.6 (la ligne vendue au depot) n'expose AUCUNE
//     API Node/Vite/Pnpm (XML du package verifie : 0 membre) ;
//  2. la ligne renommee CommunityToolkit.Aspire.Hosting.JavaScript.Extensions
//     13.x ne couvre que les monorepos Nx / Turborepo (AddNxApp,
//     AddTurborepoApp) -- un app Vite unitaire n'est ni l'un ni l'autre ;
//  3. la ligne qui porte AddViteApp (CommunityToolkit...NodeJS.Extensions
//     9.9.0, dependances >= Aspire.Hosting 9.5.2) COMPILE sous le SDK 13.4.6
//     mais sa ressource NodeAppResource 9.x n'est JAMAIS DEMARREE par
//     l'orchestrateur 13.x : apphost vert, dashboard pret, zero processus
//     pnpm/vitespawn, zero evenement de ressource (


=== fichiers du frontend (hors node_modules) ===


  frontend\index.html


  frontend\package.json


  frontend\pnpm-lock.yaml


  frontend\pnpm-workspace.yaml


  frontend\vite.config.js


  frontend\src\main.js


### Anatomie — et pourquoi `AddExecutable` plutôt que `AddViteApp`

La déclaration qui orchestre tient en deux gestes :

```csharp
var pnpmCmd = Path.Combine(
    Environment.GetFolderPath(Environment.SpecialFolder.ApplicationData), "npm", "pnpm.cmd");
builder.AddExecutable("frontend", pnpmCmd, "./frontend", "dev")
    .WithHttpEndpoint(targetPort: 5173, name: "http");
```

- **`AddExecutable(nom, commande, répertoire, args...)`** : la primitive boxed d'Aspire.Hosting 13.4.6 — lance un vrai processus et l'ajoute au modèle de ressources.
- **`WithHttpEndpoint(targetPort: 5173)`** sans port hôte : une ressource *exécutable* non conteneur ne peut pas être proxée avec `port == targetPort` (erreur mesurée du contrôle DCP : *« Non-container resources cannot be proxied when both TargetPort and Port are specified with the same value »*). Le dev-server écoute donc **directement** 5173 — la sonde du carnet est déterministe, même forme que le gateway 30000 du volet [Orleans/02](../Orleans/02-Orleans-Aspire-CoHost.ipynb).
- **Le chemin absolu de `pnpm.cmd`** n'est pas un détail : l'orchestrateur résout la commande **sans shell** — mesuré au §3.

**Le détour `AddViteApp`** — la ligne d'API qui *semble* dédiée ne l'est pas (paysage mesuré à la rédaction, NuGet + runtime) :

| Paquet | Ce qu'il expose | Verdict mesuré |
|---|---|---|
| `Aspire.Hosting` 13.4.6 (boxed) | **aucune** API Node/Vite/Pnpm (XML du package vérifié) | n/a |
| `CommunityToolkit.Aspire.Hosting.JavaScript.Extensions` 13.x | `AddNxApp`, `AddTurborepoApp` — monorepos Nx/Turborepo | un app Vite unitaire n'est ni l'un ni l'autre |
| `CommunityToolkit.Aspire.Hosting.NodeJS.Extensions` 9.9.0 | `AddViteApp`, `WithPnpm` | **compile** sous le SDK 13.4.6 mais sa ressource 9.x n'est **jamais démarrée** par l'orchestrateur 13.x (apphost vert, dashboard prêt, zéro processus pnpm/vite, zéro événement de ressource — deux lancers) |

La ligne `NodeJS.Extensions` dépend de `Aspire.Hosting.NodeJS`, arrêté à 9.5.2 : c'est l'API d'une génération d'Aspire révolue. La forme native 13.x pour un dev-server frontend est donc la primitive `AddExecutable`. L'écart pédagogique `AddViteApp` ↔ `AddExecutable` est assumé : la primitive rend **explicites** le script (`dev`), le gestionnaire et l'endpoint que l'API d'ancienne génération cachait.

## 2. Installer les dépendances du frontend

Le dépôt ne versionne pas `node_modules/` : le carnet installe les dépendances réelles du frontend avant d'orchestrer. C'est l'équivalent frontend du `dotnet restore` — et c'est bien pnpm (le gestionnaire déclaré par l'axe, graphe verrouillé par `pnpm-lock.yaml`) qui fait le travail.

In [5]:
// Installation reelle des dependances (pnpm) -- duree et code de sortie mesures.
var chrono = System.Diagnostics.Stopwatch.StartNew();
var (rcInstall, installOut) = LabShell.RunEx(LabShell.PnpmCmd, "install", LabShell.FrontendDir, 300);
chrono.Stop();
Console.WriteLine(installOut.Trim());
Console.WriteLine($"\n[code de sortie] {rcInstall} (0 = install reussi)");
Console.WriteLine($"[duree] {chrono.Elapsed.TotalSeconds:0.0} s");
var cheminVite = Path.Combine(LabShell.FrontendDir, "node_modules", "vite", "package.json");
Console.WriteLine($"[vite] {(File.Exists(cheminVite) ? Regex.Match(File.ReadAllText(cheminVite), "\"version\": \"([^\"]+)\"").Groups[1].Value : "ABSENTE -- install en echec")}");

Already up to date
Done in 10ms using pnpm v12.15.0



[code de sortie] 0 (0 = install reussi)


[duree] 0,1 s


[vite] 7.3.7


### Lecture de l'installation

pnpm a résolu le graphe depuis `pnpm-lock.yaml` et posé `node_modules/`. Le **code de sortie à 0** est la preuve que l'installation est complète — un processus qui rend la main ne dit rien à lui seul ; sur un clone neuf sans cache, la durée monte à quelques dizaines de secondes.

Réserve de mesure : dans **cette** exécution, `node_modules/` était déjà présent et l'install s'est terminée en `Already up to date` / `Done in 10ms` — un no-op, qui prouve la cohérence du lockfile mais **pas** une installation depuis zéro. La durée d'un clone neuf n'est donc pas mesurée ici.

Un réglage mérite un mot : `pnpm-workspace.yaml` porte la table `allowBuilds` (`esbuild: true`). pnpm bloque par défaut les scripts d'installation des dépendances ; esbuild en a besoin pour choisir le binaire de sa plateforme, et Vite en dépend. Le réglage n'est ni dans `package.json` (le champ `pnpm` n'y est plus lu) ni sous forme de liste : la forme retenue sur pnpm 12.15.0 est une **table** paquet → autorisation. Sans cette entrée, sur un clone dépourvu de `node_modules/`, pnpm échoue avec `ERR_PNPM_IGNORED_BUILDS` — comportement **documenté** par pnpm, **non rejoué** dans l'exécution committée (le no-op ci-dessus ne traverse pas ce chemin).

In [6]:
// Demarrage orchestre : aspire run --detach rend la main une fois l'AppHost pret.
using System.Text.RegularExpressions;
try { LabShell.RunAspire("stop", 60); } catch { /* rien ne tournait : ignorer */ }
var chronoStart = System.Diagnostics.Stopwatch.StartNew();
var sortieAspire = LabShell.RunAspire(
    $"run --detach --apphost \"{Path.Combine(LabShell.LabDir, "apphost.cs")}\"", 150);
chronoStart.Stop();
var urlDashboard = Regex.Match(sortieAspire, @"https://localhost:(\d+)/login").Groups[1].Value;
// La ligne "Logs:" porte un chemin Windows nu (mesure 2026-10-24), pas une URI
// file:/// : le motif accepte les deux formes, un poste Linux rendrait l'autre.
var mJournal = Regex.Match(sortieAspire, @"(?:file:///)?((?:[A-Za-z]:[/\\]|/)[^\s]*\.log)");
LabShell.LabLogPath = mJournal.Success ? mJournal.Groups[1].Value : "";
var vitePret = LabShell.WaitForPort(5173, 90);
Console.WriteLine(sortieAspire.Split('\n').Where(l => l.Trim().Length > 0).TakeLast(6).ToArray()
    .Select(l => Regex.Replace(l, @"\?t=[0-9a-f]+", "?t=<jeton local omis>"))
    .Select(l => Regex.Replace(l, @"[A-Za-z]:[/\\]\S*", m => m.Value.Contains(".aspire") ? "<home>" : m.Value))
    .Select(l => l.TrimEnd()).Aggregate((a, b) => a + "\n" + b));
Console.WriteLine($"\n[dev-server] ecoute sur 5173 : {vitePret} (run + attente : {chronoStart.Elapsed.TotalSeconds:0.0} s)");
Console.WriteLine($"[dashboard] https://localhost:{urlDashboard} (URL sans le jeton de session local)");

Starting Aspire AppHost in the background...
     AppHost:  apphost.cs
   Dashboard:  https://localhost:17193/login?t=<jeton local omis>
        Logs:  <home>
         PID:  32408
✅ AppHost started successfully.



[dev-server] ecoute sur 5173 : True (run + attente : 9,1 s)


[dashboard] https://localhost:17193 (URL sans le jeton de session local)


### Lecture du démarrage

L'orchestrateur a fait trois choses qu'aucun script du carnet n'a eu à faire : **lancer** le processus pnpm → vite dans le répertoire du frontend, **l'enregistrer** comme ressource nommée `frontend` au dashboard, et **exposer** son endpoint. Le `True` de la sonde TCP dit que le port 5173 — celui que l'AppHost a déclaré — écoute : la déclaration a eu un effet réel.

In [7]:
// Sondes HTTP reelles sur le dev-server orchestre.
var http = new HttpClient();
foreach (var chemin in new[] { "/", "/src/main.js", "/@vite/client" })
{
    var rep = await http.GetAsync($"http://127.0.0.1:5173{chemin}");
    var corps = await rep.Content.ReadAsStringAsync();
    var extrait = Regex.Replace(corps, @"\s+", " ");
    Console.WriteLine($"GET {chemin,-15} -> {(int)rep.StatusCode} {rep.Content.Headers.ContentType?.MediaType ?? "?"} ({corps.Length} chars)");
    if (chemin == "/")
        Console.WriteLine($"    titre : {Regex.Match(corps, "<title>(.*?)</title>").Groups[1].Value}");
    else
        Console.WriteLine($"    debut : {extrait.Substring(0, Math.Min(90, extrait.Length))}");
}

GET /               -> 200 text/html (386 chars)


    titre : FrontendLab -- ressource orchestree par Aspire


GET /src/main.js    -> 200 text/javascript (4222 chars)


    debut : // Frontend minimal : le grain porte l'orchestration Aspire, pas la richesse du // front. 


GET /@vite/client   -> 200 text/javascript (179166 chars)


    debut : import "/node_modules/.pnpm/vite@7.3.7/node_modules/vite/dist/client/env.mjs"; //#region \


### Interprétation — un dev-server Vite, pas un serveur statique

Trois preuves dans la sortie :

1. **`GET /` → 200, `text/html`** avec le `<title>` du lab : la page d'entrée est servie.
2. **`GET /src/main.js` → 200, `text/javascript`** : Vite sert le module **à la demande** — c'est le comportement du dev-server (transformation à la volée), pas d'un dossier de build figé.
3. **`GET /@vite/client` → 200** : le *client HMR*, injecté dans le HTML (`<script type="module" src="/@vite/client">` visible dans le corps de `/`). Ce chemin n'existe que dans un vrai dev-server Vite — aucun serveur statique ne le sert.

Ces trois codes 200 viennent du processus que **l'AppHost a lancé** : chaque requête transite par le port 5173 que `WithHttpEndpoint(targetPort: 5173)` a déclaré.

In [8]:
// Preuve d'orchestration : les evenements de l'AppHost (journal de la CLI Aspire).
// Le journal est tenu OUVERT par le processus de l'AppHost : File.ReadAllText
// echoue en IOException (partage demande trop strict) -- on ouvre donc en
// FileShare.ReadWrite, le meme partage qu'un tail sur un fichier vivant.
if (LabShell.LabLogPath.Length == 0 || !File.Exists(LabShell.LabLogPath))
{
    Console.WriteLine("[journal] chemin non lu dans la sortie du run -- evenements non relus");
}
else
{
    string journal;
    using (var flux = new FileStream(LabShell.LabLogPath, FileMode.Open, FileAccess.Read, FileShare.ReadWrite))
    using (var lecteur = new StreamReader(flux))
    {
        journal = lecteur.ReadToEnd();
    }
    var evenements = journal.Split('\n')
        .Where(l => l.Contains("[INFO]") && l.Contains("frontend"))
        .Where(l => l.Contains("changed state") || l.Contains("Starting process") || l.Contains("Failed to start"))
        .Select(l => Regex.Replace(l, @"^.*?\[AppHost\]\s*", ""))
        .Select(l => Regex.Replace(l, @"[A-Za-z]:[/\\][^\s""]*", m => m.Value.Contains("npm") ? "<home>/pnpm.cmd" : "<chemin>"));
    foreach (var e in evenements) Console.WriteLine(e.Trim());
}

Resource frontend/frontend-dswqgtyy changed state: Starting


Starting process...	{"Executable": "/frontend-dswqgtyy", "Reconciliation": 3, "Cmd": "<home>/pnpm.cmd", "Args": ["dev"]}


Resource frontend/frontend-dswqgtyy changed state: Running


### Interprétation — ce que la déclaration a acheté, et le piège qu'elle a résolu

1. **`Starting → Running`** : la ressource `frontend` a le même cycle de vie que n'importe quelle ressource de l'AppHost — elle vit au dashboard, elle est arrêtée par l'orchestrateur (§5).
2. **`"Args": ["dev"]`** : c'est le script `dev` de `package.json` (`vite`) qui tourne, lancé par pnpm — le gestionnaire déclaré par l'axe, pas un appel node ad hoc.
3. **Le piège du chemin, mesuré** : la première version du lab passait `"pnpm"` puis `"pnpm.cmd"` comme commande. Deux échecs réels, lus dans ce même journal : `exec: "pnpm": executable file not found` puis `exec: "pnpm.cmd": executable file not found in %PATH%`. L'orchestrateur (DCP, en Go) résout la commande **sans shell** : ni les alias, ni le répertoire `%APPDATA%\npm` hors PATH du processus ne sont explorés. La ressource passait `FailedToStart` **silencieusement** — l'apphost restait vert, le dashboard démarrait, seul le port 5173 restait muet. D'où le chemin absolu calculé au runtime dans `apphost.cs` (`%APPDATA%\npm\pnpm.cmd`), même forme que le précédent `GenAiStack.AppHost` (`UserProfile` + cache HuggingFace) : le source reste portable, la résolution est explicite.

C'est la leçon transversale du volet : **une ressource qui ne démarre pas ne fait pas échouer l'AppHost** — l'échec vit dans les événements de ressources, pas dans le code de sortie du run.

## 3. Parité — orchestration frontend : AppHost C# vs `docker-compose` Python

La pile GenAI Python orchestre ses services par `docker-compose` (fichiers YAML de `docker-configurations/`). Même besoin, deux formes :

| | Aspire AppHost (ce lab) | docker-compose (pile GenAI) |
|---|---|---|
| **Déclaration** | C# typé : `AddExecutable("frontend", pnpmCmd, "./frontend", "dev")` | YAML : `build: ./frontend`, `command: pnpm dev` |
| **Cycle de vie** | Démarré/arrêté **avec l'AppHost** (`aspire run` / `aspire stop`) — le frontend meurt avec sa pile | `docker compose up/down` — un arbre de conteneurs séparé du reste de la stack |
| **Observabilité** | Ressource nommée au **dashboard unifié** (logs, événements `Starting/Running`) à côté des backends | `docker logs` par conteneur, pas de vue unifiée avec un backend .NET |
| **Dev-server vs image** | **Dev-server** : HMR, transformation à la volée, pas de build | Image buildée : re-build à chaque itération du front |
| **Isolation par worktree** | Chaque clone a son AppHost et ses ports | Ports fixes par fichier compose — collision entre worktrees |
| **Coût** | Écrire la résolution du gestionnaire à la main (chemin, §interprétation précédente) | Écrire un Dockerfile de plus |

Le choix du lab — dev-server orchestré plutôt qu'image — est un choix **de boucle locale** : en production, le frontend buildé (`pnpm build`) est servi par un backend statique ou un CDN, pas par Vite. Le volet garde les deux histoires séparées.

In [9]:
// Garde SOTA : ce qui a servi les reponses est le vrai dev-server Vite, pas un substitut.
var http2 = new HttpClient();
var repHmr = await http2.GetAsync("http://127.0.0.1:5173/@vite/client");
var corpsHmr = await repHmr.Content.ReadAsStringAsync();
bool hmrPing = corpsHmr.Contains("ping"), hmrWs = corpsHmr.Contains("WebSocket");
Console.WriteLine($"[@vite/client] {(int)repHmr.StatusCode}, marqueurs du client HMR : ping={hmrPing}, WebSocket={hmrWs}");
var pkgVite = File.ReadAllText(Path.Combine(LabShell.FrontendDir, "node_modules", "vite", "package.json"));
Console.WriteLine($"[vite reel] version installee {Regex.Match(pkgVite, "\"version\": \"([^\"]+)\"").Groups[1].Value}");
Console.WriteLine("[verdict] SOTA-OK : dev-server Vite reel, pnpm reel, AppHost reel -- aucune reimplementation");

[@vite/client] 200, marqueurs du client HMR : ping=True, WebSocket=True


[vite reel] version installee 7.3.7


[verdict] SOTA-OK : dev-server Vite reel, pnpm reel, AppHost reel -- aucune reimplementation


### Garde SOTA — verdict

`/@vite/client` rend le code du client HMR : un serveur de substitution n'aurait ni `ping`, ni `WebSocket`. Le verdict **SOTA-OK** (outil réel installé et invoqué, aucun workaround dégradé) est mesuré sur ces marqueurs — pas décrété.

## 4. Exercices

Les trois exercices se font **sans changer l'orchestration** : le frontend est une ressource déclarée, on itère sur ses fichiers. Après chaque modification, relancer la cellule de démarrage (§2) — `aspire stop` puis `run --detach` — pour que l'AppHost reprenne l'état du disque.

Chaque énoncé a sa cellule **témoin** : elle s'exécute telle quelle et affiche le résultat attendu **tant que l'exercice n'est pas complété** (règle C.1 : aucune erreur volontaire — le carnet s'exécute de bout en bout).

### Exercice 1 — une page servie par le dev-server

Vite sert une **application monopage** : une requête HTML inconnue (`/about.html`) ne rend pas 404, elle rend `index.html` en **200**. Le code de statut ne prouve donc rien ici — c'est le **contenu** qui distingue une page créée d'un repli.

Étapes :
1. Créer `FrontendLab/frontend/about.html` : un HTML minimal portant `<div id="about-page">`.
2. Relancer la cellule de démarrage (§2).
3. Relire la sonde ci-dessous : elle cherche le marqueur `id="about-page"` dans la réponse.

`# TODO étudiant` : le témoin doit passer de `False` à `True` — le statut, lui, reste 200 dans les deux cas.

In [10]:
// Exercice 1 -- temoin sur le CONTENU : Vite rend index.html en 200 pour une page
// inconnue (repli SPA, mesure), donc le statut ne distingue pas une page creee
// d'un repli. Le marqueur, lui, n'apparait que dans la page de l'etudiant.
var httpE1 = new HttpClient();
var repE1 = await httpE1.GetAsync("http://127.0.0.1:5173/about.html");
var corpsE1 = await repE1.Content.ReadAsStringAsync();
var pageE1 = corpsE1.Contains("id=\"about-page\"");
Console.WriteLine($"GET /about.html -> {(int)repE1.StatusCode}, marqueur id=\"about-page\" : {(pageE1 ? "present" : "absent")}");
Console.WriteLine($"[temoin] page dediee servie : {pageE1} (attendu False tant que l'exercice n'est pas complete)");
Console.WriteLine("Exercice a completer : creer FrontendLab/frontend/about.html avec <div id=\"about-page\">, puis relancer le demarrage.");

GET /about.html -> 200, marqueur id="about-page" : absent


[temoin] page dediee servie : False (attendu False tant que l'exercice n'est pas complete)


Exercice a completer : creer FrontendLab/frontend/about.html avec <div id="about-page">, puis relancer le demarrage.


### Exercice 2 — déplacer le port déclaré

L'endpoint vit en **deux** endroits qui doivent rester cohérents : `vite.config.js` (ce que le dev-server écoute) et `apphost.cs` (ce que l'AppHost déclare).

Étapes :
1. Dans `vite.config.js` : `port: 5175`.
2. Dans `apphost.cs` : `WithHttpEndpoint(targetPort: 5175, ...)`.
3. Relancer la cellule de démarrage, puis relire le témoin.

`# TODO étudiant` : le témoin doit passer de `False` à `True`.

In [11]:
// Exercice 2 -- temoin attendu tant que le port n'est pas deplace : 5175 muet.
var portE2 = 5175;
var pretE2 = LabShell.WaitForPort(portE2, 3);
Console.WriteLine($"[port {portE2}] ecoute : {pretE2} (attendu False tant que l'exercice n'est pas complete)");
Console.WriteLine("Exercice a completer : deplacer le port dans vite.config.js ET apphost.cs, puis relancer le demarrage.");

[port 5175] ecoute : False (attendu False tant que l'exercice n'est pas complete)


Exercice a completer : deplacer le port dans vite.config.js ET apphost.cs, puis relancer le demarrage.


### Exercice 3 — un point d'API simulé côté client

Le lab n'a pas de backend : l'exercice ajoute au `src/main.js` un rendu de « lecture simulée » — une donnée calculée côté client, affichée dans la page.

Étapes :
1. Dans `src/main.js` : ajouter au plateau une ligne `apis: ['GET /api/time (simule)']` alimentée par `new Date().toLocaleTimeString('fr-FR')`.
2. Relancer la cellule de démarrage.
3. Relire le témoin : la source servie doit contenir la nouvelle ligne.

`# TODO étudiant` : le témoin doit passer de `absent` à `present`.

In [12]:
// Exercice 3 -- temoin attendu tant que la ligne api n'est pas ajoutee : absent.
var httpE3 = new HttpClient();
var repE3 = await httpE3.GetAsync("http://127.0.0.1:5173/src/main.js");
var srcE3 = await repE3.Content.ReadAsStringAsync();
var presentE3 = srcE3.Contains("/api/time");
Console.WriteLine($"[src/main.js] ligne /api/time : {(presentE3 ? "present" : "absent")} (attendu absent tant que l'exercice n'est pas complete)");
Console.WriteLine("Exercice a completer : ajouter la lecture simulee dans src/main.js puis relancer le demarrage.");

[src/main.js] ligne /api/time : absent (attendu absent tant que l'exercice n'est pas complete)


Exercice a completer : ajouter la lecture simulee dans src/main.js puis relancer le demarrage.


## 5. Arrêter proprement

Dernier geste d'orchestration : `aspire stop` coupe l'application distribuée **entière** — le dev-server ET le dashboard — d'un seul appel. C'est la réciproque du démarrage : ce que l'AppHost a lancé, l'AppHost l'arrête.

In [13]:
// Arret orchestre : aspire stop coupe l'AppHost et tout ce qu'il a lance.
var stopOut = LabShell.RunAspire("stop", 60);
Console.WriteLine(string.Join("\n", stopOut.Split('\n').Where(l => l.Trim().Length > 0).TakeLast(4)));
Thread.Sleep(2000);
var encoreLa = LabShell.WaitForPort(5173, 2);
Console.WriteLine($"\n[dev-server] ecoute encore sur 5173 : {encoreLa} (attendu False)");

📦 Found running AppHost: apphost.cs
🛑 Sending stop signal to apphost.cs...
Stopping apphost.cs...




[dev-server] ecoute encore sur 5173 : False (attendu False)


## Ce que ce notebook ne couvre pas (limites honnêtes)

- **Pas de proxy `/api` vers un backend .NET** : le lab orchestre le frontend **seul**. Le chaînage frontend → backend (une ressource `project` .NET + `WithReference`) est le grain suivant naturel — l'exercice 3 n'en est qu'une simulation côté client.
- **Dev-server, pas la production** : le lab orchestre `vite` (dev). Le build de production (`pnpm build` → `dist/`) se sert hors Vite — il n'est pas orchestré ici.
- **Windows mesuré** : le chemin `pnpm.cmd` et le wrapper `cmd.exe` sont les formes Windows du lab ; sur Linux/macOS, il faudrait adapter la résolution (`pnpm` depuis le PATH ou `~/.local/share/pnpm`, sans suffixe `.cmd`) — le code ne le fait pas encore, aucune branche d'OS n'y figure. Le carnet est exécuté et committé depuis un poste Windows.
- **L'API d'ancienne génération (`AddViteApp`)** : le paysage mesuré au §1 est daté — si la CommunityToolkit ressuscite la ligne `NodeJS`, la déclaration pourra s'y rabattre ; la mesure sera à refaire.

**Registre** : ce volet solde l'axe « AppHost frontend » différé par [#10897](https://github.com/jsboige/CoursIA/issues/10897) — ligne mise à jour dans [`distilled-axes-registry.md`](distilled-axes-registry.md).